# Scam Shield: fine-tune the text head on Kaggle

Runs `src/finetune.py` from the private repo on a Kaggle GPU (T4 or P100). This notebook only clones and runs; never edit code here (CLAUDE.md).

Before running:
- **Settings → Accelerator:** a GPU (T4 x2 or P100). **Internet:** on (clone + Hugging Face download).
- **Add-ons → Secrets:** `GH_TOKEN`, a fine-grained GitHub token with read-only *Contents* access to `bored-psychic/scam-shield`, attached to this notebook.
- **Input:** a **private** Kaggle dataset `scam-shield-data` with a folder `final/` holding `split_map.csv`, `messages.csv`, `messages_adv_test.csv` (+ `messages_aug.csv` for `--aug`). `finetune.py` refuses a `split_map.csv` that isn't splits-v1.

Precision is picked by `finetune.py --precision auto`: T4 (7.5) and P100 (6.0) get fp16 + GradScaler; bf16 needs compute capability >= 8.0.

In [ ]:
# ---- settings ----------------------------------------------------------------------------------------------------
ARGS = "--model muril --smoke"                          # e.g. "--model muril", "--model mdeberta", "--model muril --aug"
import glob, os
from pathlib import Path

# find the dataset folder by its split_map.csv, wherever Kaggle mounted it (exactly one expected)
hits = sorted(glob.glob("/kaggle/input/**/split_map.csv", recursive=True))
if len(hits) != 1:
    listing = "\n  ".join(sorted(os.listdir("/kaggle/input"))) or "(empty)"
    raise RuntimeError(f"expected exactly one split_map.csv under /kaggle/input, found {len(hits)}: {hits}\n"
                       f"/kaggle/input contains:\n  {listing}\nAdd the private dataset scam-shield-data (its folder must contain split_map.csv).")
DATA_DIR = str(Path(hits[0]).parent)
print("DATA_DIR =", DATA_DIR)
REPO_URL = "https://github.com/bored-psychic/scam-shield.git"
REPO_DIR = "/kaggle/working/scam-shield"
OUT_ROOT = "/kaggle/working/models"

In [ ]:
# ---- clone (or update) the private repo; the token is never printed, never in argv, never written to .git/config -------
import base64, os, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

def git(*args, token=None, cwd=None):
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    if token:   # one-off config from the environment (git >= 2.31): an Authorization header for github.com, this call only
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")
    r = subprocess.run(["git", *args], cwd=cwd, env=env, capture_output=True, text=True)
    out = (r.stdout + r.stderr)
    if token:
        out = out.replace(token, "***").replace(basic, "***")
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out.strip()

token = UserSecretsClient().get_secret("GH_TOKEN")
try:
    if (Path(REPO_DIR) / ".git").exists():
        print(git("pull", "--ff-only", token=token, cwd=REPO_DIR))
    else:
        print(git("clone", "--quiet", REPO_URL, REPO_DIR, token=token) or f"cloned into {REPO_DIR}")
finally:
    del token

git("remote", "set-url", "origin", REPO_URL, cwd=REPO_DIR)   # plain https URL, no credentials
cfg = (Path(REPO_DIR) / ".git" / "config").read_text()
assert "@github.com" not in cfg and "extraheader" not in cfg.lower() and "x-access-token" not in cfg, "credentials in .git/config"
print("HEAD", git("rev-parse", "HEAD", cwd=REPO_DIR), "|", git("log", "-1", "--format=%s", cwd=REPO_DIR))
print("tree clean:", git("status", "--porcelain", "--untracked-files=no", cwd=REPO_DIR) == "")

In [ ]:
# ---- install only what's missing; keep Kaggle's own torch (the cu130 pin is for the RTX 5050 laptop) --------------------
import importlib.metadata as md, re, subprocess, sys

pins = {}
for line in open(f"{REPO_DIR}/requirements-finetune.txt"):
    m = re.match(r"^([A-Za-z0-9_.-]+)==(\S+)", line.strip())
    if m and m.group(1).lower() != "torch":
        pins[m.group(1)] = m.group(2)

missing, differ = [], []
for name, want in pins.items():
    try:
        have = md.version(name)
    except md.PackageNotFoundError:
        missing.append(f"{name}=={want}")
        continue
    if have != want:
        differ.append((name, have, want))
# tldextract is the exception: textprep.model_text uses its bundled suffix list, so its version changes model inputs (rule 11)
for name, have, want in differ:
    if name == "tldextract":
        missing.append(f"{name}=={want}")
if missing:
    print("installing:", " ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("nothing missing")
for name, have, want in differ:
    if name != "tldextract":
        print(f"kept Kaggle's {name} {have} (pin {want}); shield_meta.json records the version used")

In [ ]:
# ---- GPU and versions -----------------------------------------------------------------------------------------------
import subprocess, torch, transformers, tokenizers, tldextract
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| transformers", transformers.__version__,
      "| tokenizers", tokenizers.__version__, "| tldextract", tldextract.__version__)
assert torch.cuda.is_available(), "no GPU: Settings -> Accelerator"
cap = torch.cuda.get_device_capability(0)
print("GPU", torch.cuda.get_device_name(0), "| compute capability", f"{cap[0]}.{cap[1]}",
      "| auto precision ->", "bf16" if cap >= (8, 0) else "fp16 + GradScaler")
print("torch built for:", torch.cuda.get_arch_list())
# fail here, not 10 minutes in, if this torch build has no kernels for the GPU (recent wheels dropped Pascal / P100, sm_60)
x = torch.randn(256, 256, device="cuda")
with torch.autocast("cuda", dtype=torch.float16):
    y = x @ x
torch.cuda.synchronize()
print("fp16 matmul ok:", tuple(y.shape), y.dtype)

In [ ]:
# ---- train ------------------------------------------------------------------------------------------------------------
import shlex, subprocess, sys
from datetime import datetime, timezone

parts = shlex.split(ARGS)
name = parts[parts.index("--model") + 1] if "--model" in parts else "muril"
RUN_DIR = f"{OUT_ROOT}/{name.split('/')[-1]}_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}{'_smoke' if '--smoke' in parts else ''}"
cmd = [sys.executable, "-u", "src/finetune.py", *parts, "--data-dir", DATA_DIR, "--out", RUN_DIR]
print(" ".join(cmd), flush=True)
p = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="", flush=True)
assert p.wait() == 0, f"finetune.py exited with {p.returncode}"

In [ ]:
# ---- zip the run for download (without the epoch_* checkpoints; model/ is the best epoch) -----------------------------
import json, zipfile
from pathlib import Path

run = Path(RUN_DIR)
zip_path = Path("/kaggle/working") / f"{run.name}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(run.rglob("*")):
        rel = f.relative_to(run)
        if f.is_file() and not rel.parts[0].startswith("epoch_"):
            z.write(f, Path(run.name) / rel)
meta = json.loads((run / "shield_meta.json").read_text())
print(f"{zip_path} ({zip_path.stat().st_size / 1e6:.0f} MB)")
print({k: meta[k] for k in ("git_commit", "gpu_name", "precision", "best_epoch", "threshold_suspicious", "threshold_likely")})
print("Download it from the Output panel (or Save Version -> Output), then unzip into models/finetune/ on the Mac.")